In [ ]:
import subprocess
import numpy as np
from numpy import linalg as LA
import matplotlib
import json

matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

In [ ]:
def get_data(bound, count, output="output.tmp",
             scheme="euler", summation="simple", field="classic",
             binary=True, omega0=1.0, gamma=1.0, A=1.0, omega=1.0, x0=1.0, v0=1.0):
    settings = {
        "output": output,
        "scheme": scheme,
        "summation": summation,
        "field": field,
        "binary": binary,
        "bound": bound,
        "count": count,
        "omega0": omega0,
        "gamma": gamma,
        "A": A,
        "omega": omega,
        "x0": x0,
        "v0": v0
    }
    with open("settings.json", "w") as f:
        json.dump(settings, f)

    res = subprocess.run(["./solver", "settings.json"])
    if binary:
        data = np.fromfile(output, dtype="<f8").reshape(-1, 3)
        data = data.T
    else:
        data = np.loadtxt(output, delimiter=" ", dtype=float)
        data = (data[:, 0], data[:, 1], data[:, 2])
    
    return data

def lin(x, y):
    n = len(x)
    
    k = np.sum(x * y) / np.sum(x**2)
    
    residuals = y - k * x
    if n > 1:
        s2 = np.sum(residuals**2) / (n - 1)
    else:
        s2 = 0.0
    
    err_k = np.sqrt(s2 / np.sum(x**2))
    
    return k, err_k

def lin_(x, y):
    xy = x*y
    k = (np.mean(xy) - np.mean(x) * np.mean(y))/\
        (np.mean(x**2) - np.mean(x)**2)
    a = np.mean(y) - k*np.mean(x)

    sqrtn = (len(xy))**0.5
    err_k = 1/sqrtn * ((np.mean(y**2) - np.mean(y)**2)/\
                       (np.mean(x**2) - np.mean(x)**2) - k**2)**0.5
    err_a = err_k*(np.mean(x**2) - np.mean(x)**2)**0.5

    return k, err_k, a, err_a

Properties

In [ ]:
x0 = 0.5
v0 = 0.0
omega0 = 1.0
gamma = 0.1
A = 0.5
omega = 0.4439

d = omega0**2 - omega**2
q = 2 * gamma * omega
Q = d**2 + q**2
a = A*d/Q
b = A*q/Q
Af = A/np.sqrt(Q)
Om = np.sqrt(omega0**2 - gamma**2)
C = x0 - a
D = (v0 + gamma * C - omega * b)/Om

def get_analytic(omega):
    d = omega0**2 - omega**2
    q = 2 * gamma * omega
    Q = d**2 + q**2
    a = A*d/Q
    b = A*q/Q
    Af = A/np.sqrt(Q)
    Om = np.sqrt(omega0**2 - gamma**2)
    C = x0 - a
    D = (v0 + gamma * C - omega * b)/Om
    def analytic(t):
        return np.exp(-gamma * t) * (C * np.cos(Om * t) + D * np.sin(Om * t)) + \
            a * np.cos(omega * t) + b * np.sin(omega * t)

    return analytic

def get_all(bound, count, scheme, field, gamma=gamma, omega=omega, summation="simple"):
    data = get_data(bound, count, field=field, scheme=scheme, summation=summation,
                    omega0=omega0, gamma=gamma, A=A, omega=omega, x0=x0, v0=v0)
    t, x, v = data
    return t, x, v

Plot

In [ ]:
method = "rk4"
field = "phys"

bound = 500
count = 100000

t, x, v = get_all(bound, count, method, field, summation="kahan")
plt.grid()
plt.plot(t, x)
# plt.plot(t, A0 * np.exp(-gamma * t) * np.cos(Om * t) - x)
# plt.plot(t, analytic(t))
plt.xlabel("$t$")
plt.ylabel(r"$x$")
plt.show()

plt.close()

FFT

In [ ]:
method = "rk4"
field = "classic"

bound = 300
count = 100000

t, x, v = get_all(bound, count, method, field, summation="kahan")
plt.grid()
x_stable = x[t > 8/gamma]
t_stable = t[t > 8/gamma]
dt = t_stable[1] - t_stable[0]
N = len(x_stable)

X = np.fft.rfft(x_stable)
freq = np.fft.rfftfreq(N, d=dt)
amp = np.abs(X) / N
plt.plot(2 * np.pi * freq, amp)
plt.xlabel(r"$\omega$")
plt.ylabel(r"$A$")
plt.show()

plt.close()

Phase diagrams

In [ ]:
method = "rk4"
field = "phys"

bound = 100
count = 50000

t, x, v = get_all(bound, count, method, field, summation="kahan")
plt.grid()
plt.plot(x, v)
plt.xlabel("$x$")
plt.ylabel(r"$\dot{x}$")

plt.show()
plt.close()

AFR

In [ ]:
method = "rk4"
field = "classic"
w_start = 0.3
w_end = 3.0
countw = 500

w = np.linspace(w_start, w_end, countw)
def get_afr(gamma):
    bound = 300
    count = 5000

    a = []
    for wi in w:
        t, x, v = get_all(bound, count, method, field,
                        gamma=gamma,
                        omega = wi,
                        summation="kahan")
        x_after = x[t > 8 / gamma]
        N = len(x_after)
        window = np.hanning(N)
        nfft = 1 << (16 * N - 1).bit_length()
        X = np.abs(np.fft.rfft(x_after * window), n=nfft) / window.sum()
        a.append(np.max(X))

    a = np.array(a)
    return a

for gamma in [0.1]:
    a = get_afr(gamma)
    plt.plot(w, a)

plt.grid()
plt.xlabel(r"$\omega$")
plt.ylabel(r"$A$")

# plt.axhline(y=np.max(a) / np.sqrt(2), color="red", linestyle="--")
plt.show()

Bounds of methods

Find error

In [ ]:
summation = "kahan"
type = "classic"
for m in ["euler", "heun", "rk4"]:
    bound = 10.0
    dots = 30

    count = 1.25
    mult = np.sqrt(2)

    times = []
    errors = []
    for i in range(dots):
        dt = bound/count
        times.append(dt)

        t, x, v = get_all(bound, count, m, type, summation=summation)
        err = np.abs(A0 * np.exp(-gamma * t) * np.cos(Om * t) - x)
        errors.append(np.max(err))

        count *= mult

    times = np.log(times)
    errors = np.log(errors)
    k, err_k, a, err_a = lin_(times, errors)
    print(m, k)
    plt.plot(times, errors, label=m)

plt.xlabel(r"$\ln{\Delta t}$")
plt.ylabel(r"$\ln{\text{Error}}$")
plt.legend()
plt.grid()
plt.show()